# 03 · EVE — an unsupervised evolutionary model

**CFTR variant-effect toolkit · beginner track.** EVE (*Evolutionary model of Variant Effect*, Frazer et al. 2021, *Nature*, PMID 34707284) is a deep generative model of a protein family's evolutionary variation. It is **unsupervised** — it never saw a clinical label — which is what makes it fair to benchmark against ClinVar later (notebook 12).

### The EVE data file — where it comes from, and its columns

`build_eve.py` reads `variant_files/CFTR_HUMAN.csv` from the EVE release zip
(evemodel.org; CFTR = UniProt **P13569**). The **zip** also contains the multiple-
sequence alignments (`MSAs/`) and ROC/PRC curve images. `CFTR_HUMAN.csv` has **~42
columns**; the build keeps five:

| column (source) | meaning | in the extract |
|---|---|---|
| `wt_aa`, `position`, `mt_aa` | wild-type AA, residue number, mutant AA | → `protein_variant` (e.g. `M1A`) |
| `EVE_scores_ASM` | EVE pathogenicity, 0–1 (higher = more pathogenic) | → `eve_score` |
| `EVE_classes_75_pct_retained_ASM` | Benign/Pathogenic call at the threshold that leaves the least-confident **25% as "Uncertain"** | → `eve_class` |

The other ~37 columns (not used here) include `evolutionary_index_ASM`, `uncertainty_ASM`,
EVE classes at every retention threshold from 10–90%, a separate **BPU** model, and
ClinVar / frequency / ACMG-style flag columns — see the EVE docs.

**Coverage caveat:** EVE is **not** full saturation. It scores only alignable positions
(limited by alignment depth), so the ~26,809 scored variants cover a *subset* of the
1,480 × 19 possible — unlike ESM1b (notebook 04), which is saturation.

> ✅ **REAL DATA.** This notebook uses the real EVE scores for CFTR (**~26,809** scored variants, evemodel.org / UniProt P13569), extracted by `build_eve.py` → `data/eve_cftr_2021-08.csv` and returned by `tk.load_eve()`. Every table's `source` column reads `REAL`. *(EVE code is MIT; confirm the score-redistribution terms on evemodel.org before publishing the extract.)*

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · EVE — an evolutionary generative model

**What is it?** EVE (*Evolutionary model of Variant Effect*) is a deep **generative model** trained on the **multiple-sequence alignment (MSA)** of a protein family — i.e. the same protein lined up across hundreds or thousands of species. Evolution has already run a giant natural experiment: positions that *must not change* to keep the protein working stay constant across species, while positions that tolerate change vary freely.

EVE learns that pattern of *evolutionary constraint*, then scores a new variant by asking: **"how well does this amino-acid change fit what evolution has tolerated here?"** A change at a rigidly-conserved position looks evolutionarily *implausible* → high score.

- **Score range:** `[0, 1]` (a posterior probability of being pathogenic).
- **Rule of thumb:** `>= 0.5` ~ pathogenic, `< 0.5` ~ benign. Higher = more damaging.
- **Unsupervised:** trained only on sequences, *never* on ClinVar labels → **low circularity** when we later benchmark it against clinical databases.

> ### 📥 REAL EVE data (already wired in)
> This notebook uses the real per-protein EVE scores for **CFTR (UniProt `P13569`)** from **https://evemodel.org**, extracted by `build_eve.py` → `data/eve_cftr_2021-08.csv` and returned by `tk.load_eve()`. To refresh, drop a newer EVE release in and re-run the builder.

> **EVE resources:** paper Frazer et al. 2021, *Nature*, PMID 34707284; scores at **evemodel.org**; open-source code (MIT) at **github.com/OATML-Markslab/EVE**.

In [2]:
eve = tk.load_eve()      # REAL — CFTR EVE extract (build_eve.py from evemodel.org)
print(f'{len(eve)} REAL EVE variants | source: {eve["source"].unique().tolist()}')
print(f'residues covered: {eve["protein_variant"].str.extract(r"(\\d+)")[0].nunique()}')
print()
print(eve['eve_class'].value_counts().to_string())
eve.head(8)

26809 REAL EVE variants | source: ['REAL']
residues covered: 0

eve_class
Pathogenic    15879
Uncertain      5536
Benign         5394


,protein_variant,eve_score,eve_class,source
0,L15A,0.520237,Uncertain,REAL
1,L15C,0.600187,Uncertain,REAL
2,L15D,0.831544,Pathogenic,REAL
3,L15E,0.579939,Uncertain,REAL
4,L15F,0.237468,Benign,REAL
5,L15G,0.625586,Uncertain,REAL
6,L15H,0.625182,Uncertain,REAL
7,L15I,0.177613,Benign,REAL


## 2 · Turn an EVE score into a call

`tk.call_from_score(score, 'eve')` applies EVE's published threshold (`>= 0.5` ~ pathogenic, higher = worse) and returns `pathogenic / uncertain / benign`.

In [3]:
eve['eve_call'] = eve['eve_score'].apply(lambda s: tk.call_from_score(s, 'eve'))
eve[['protein_variant', 'eve_score', 'eve_class', 'eve_call', 'source']].head(12)

,protein_variant,eve_score,eve_class,eve_call,source
0,L15A,0.520237,Uncertain,pathogenic,REAL
1,L15C,0.600187,Uncertain,pathogenic,REAL
2,L15D,0.831544,Pathogenic,pathogenic,REAL
3,L15E,0.579939,Uncertain,pathogenic,REAL
4,L15F,0.237468,Benign,benign,REAL
5,L15G,0.625586,Uncertain,pathogenic,REAL
6,L15H,0.625182,Uncertain,pathogenic,REAL
7,L15I,0.177613,Benign,benign,REAL
8,L15K,0.599925,Uncertain,pathogenic,REAL
9,L15M,0.201823,Benign,benign,REAL


## 3 · The A1 "Priority-1" VUS now have REAL EVE scores

The curated A1 Priority-1 VUS were keyed 3-letter (`Tyr161Cys`); real EVE uses 1-letter (`Y161C`). `tk.three_to_one()` bridges the two — and the real numbers are sobering.

In [4]:
p1 = ['Tyr161Cys', 'Gly970Asp', 'Ser912Leu', 'Val520Phe']
rows = []
for v in p1:
    k = tk.three_to_one(v)
    hit = eve[eve['protein_variant'] == k]
    rows.append({'demo_key': v, 'eve_key': k,
                 'real_eve': None if hit.empty else round(float(hit['eve_score'].iloc[0]), 3),
                 'real_eve_call': None if hit.empty else tk.call_from_score(hit['eve_score'].iloc[0], 'eve'),
                 'real_eve_class': None if hit.empty else hit['eve_class'].iloc[0]})
pd.DataFrame(rows)

,demo_key,eve_key,real_eve,real_eve_call,real_eve_class
0,Tyr161Cys,Y161C,0.934,pathogenic,Pathogenic
1,Gly970Asp,G970D,0.930,pathogenic,Pathogenic
2,Ser912Leu,S912L,0.085,benign,Benign
3,Val520Phe,V520F,0.948,pathogenic,Pathogenic


Note **Ser912Leu → S912L → EVE ≈ 0.085 (benign)** — real EVE *disagrees* with this variant's CF-causing clinical evidence (a known EVE false negative; McDonald et al. 2023). The hand-authored demo score (0.742) had hidden that. This is exactly why REAL data + honest provenance matter — and why a single predictor, even a real one, must be benchmarked against orthogonal truth (notebook 12).

## Key takeaways

1. **EVE** models a protein family's **evolutionary** constraint (its MSA). Score `[0,1]`, `>= 0.5` ~ pathogenic, **higher = worse**.
2. **Unsupervised** w.r.t. clinical labels → **low circularity** vs ClinVar.
3. This notebook now uses **REAL EVE** — ~26,809 CFTR variants (evemodel.org / P13569), shipped as `data/eve_cftr_2021-08.csv`; `source` reads `REAL`.
4. Real EVE covers the A1 Priority-1 VUS (via `three_to_one`) and, e.g., calls **S912L benign** — a reminder that even a real predictor can be wrong; benchmark against orthogonal truth (notebook 12).

**Next:** notebook 04 — **ESM1b**, a protein language model whose scale runs *backwards*. The EVE-vs-ESM1b comparison lives in the archived integration notebook.